# Scene Weaver — encoder (Google Colab, GPU **or** CPU)

Works on any runtime. T4 GPU = fastest (NVENC); CPU-only runtimes fall back to libx264 automatically.

1. `Runtime → Run all`
2. Copy the **https link** printed at the bottom into the app's *Colab encoder* box.

Keep this tab open while the video renders.


In [ ]:
#@title 1 · Start encoder + install tunnel automatically
APP_URL = "https://epic-colab-maker.lovable.app"  #@param {type:"string"}
TOKEN   = ""  #@param {type:"string"}

import os, re, base64, subprocess, threading, time, urllib.request, shutil, platform
os.environ['SW_TOKEN'] = TOKEN

# download latest encoder from the live app; fall back to the embedded copy
try:
    req = urllib.request.Request(APP_URL.rstrip('/') + '/colab/encoder_server.py',
                                 headers={'User-Agent': 'scene-weaver-colab'})
    with urllib.request.urlopen(req, timeout=30) as r, open('/content/encoder_server.py', 'wb') as f:
        f.write(r.read())
    print('encoder downloaded from', APP_URL)
except Exception as e:
    print(f'download failed ({e}) — using embedded encoder copy')
    _B64 = "IiIiClNjZW5lIFdlYXZlciDigJQgQ29sYWIgVDQgR1BVIGVuY29kZXIuCgpSdW5zIGluc2lkZSBhIEdvb2dsZSBDb2xhYiBub3RlYm9vayAoR1BVIHJ1bnRpbWUpLiBBY2NlcHRzIGEgcGFuZWwgbGlzdCBmcm9tCnRoZSB3ZWIgYXBwLCBkb3dubG9hZHMgZXZlcnkgcGFuZWwgaW1hZ2UsIHJlbmRlcnMgS2VuIEJ1cm5zIG1vdGlvbiArIGNvbG91cgpncmFkaW5nIHdpdGggZmZtcGVnLCBlbmNvZGVzIHdpdGggdGhlIFQ0J3MgTlZFTkMgaGFyZHdhcmUgZW5jb2RlciBhbmQgc2VydmVzCnRoZSBmaW5pc2hlZCBtcDQgYmFjayBvdmVyIGFuIGh0dHBzIHR1bm5lbC4KCkRlc2lnbiBub3RlcyBmb3IgdmVyeSBsb25nIHZpZGVvcyAoMmgrLCB0aG91c2FuZHMgb2YgcGFuZWxzKToKICAqIEVhY2ggcGFuZWwgYmVjb21lcyBpdHMgb3duIHNob3J0IGNsaXAgLT4gbWVtb3J5IHN0YXlzIGZsYXQuCiAgKiBDbGlwcyBhcmUgY3Jvc3MtZmFkZWQgaW4gZ3JvdXBzIChHUk9VUCBwYW5lbHMgcGVyIGZpbHRlcl9jb21wbGV4KSBzbyB0aGUKICAgIGZmbXBlZyBjb21tYW5kIG5ldmVyIGdyb3dzIHVuYm91bmRlZCwgdGhlbiB0aGUgZ3JvdXBzIGFyZSBzdHJlYW0tY29weQogICAgY29uY2F0ZW5hdGVkOiBubyBnZW5lcmF0aW9uIGxvc3MsIG5vIE8obl4yKSByZS1lbmNvZGluZy4KICAqIFBhbmVscyBhcmUgcmVuZGVyZWQgaW4gcGFyYWxsZWwgbGFuZXM7IE5WRU5DIG9uIGEgVDQgaGFuZGxlcyBzZXZlcmFsCiAgICAxMDgwcDMwIHN0cmVhbXMgYXQgb25jZS4KIiIiCgppbXBvcnQganNvbiwgbWF0aCwgb3MsIHJlLCBzaHV0aWwsIHN1YnByb2Nlc3MsIHRocmVhZGluZywgdGltZSwgdXVpZCwgaGFzaGxpYgpmcm9tIGNvbmN1cnJlbnQuZnV0dXJlcyBpbXBvcnQgVGhyZWFkUG9vbEV4ZWN1dG9yCmZyb20gaHR0cC5zZXJ2ZXIgaW1wb3J0IEJhc2VIVFRQUmVxdWVzdEhhbmRsZXIsIFRocmVhZGluZ0hUVFBTZXJ2ZXIKZnJvbSB1cmxsaWIucGFyc2UgaW1wb3J0IHVybHBhcnNlCmltcG9ydCB1cmxsaWIucmVxdWVzdAoKVywgSCwgRlBTLCBYRiwgR1JPVVAgPSAxOTIwLCAxMDgwLCAzMCwgMC43LCA0MApXT1JLID0gIi9jb250ZW50L3N3X3dvcmsiCk9VVCA9ICIvY29udGVudC9zd19vdXQiCkxBTkVTID0gaW50KG9zLmVudmlyb24uZ2V0KCJTV19MQU5FUyIsICIwIikpClRPS0VOID0gb3MuZW52aXJvbi5nZXQoIlNXX1RPS0VOIiwgIiIpCgpvcy5tYWtlZGlycyhXT1JLLCBleGlzdF9vaz1UcnVlKQpvcy5tYWtlZGlycyhPVVQsIGV4aXN0X29rPVRydWUpCgpKT0JTID0ge30KTE9DSyA9IHRocmVhZGluZy5Mb2NrKCkKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBlbmNvZGVyIC0tLQoKZGVmIGhhc19udmVuYygpOgogICAgdHJ5OgogICAgICAgIG91dCA9IHN1YnByb2Nlc3MucnVuKFsiZmZtcGVnIiwgIi1oaWRlX2Jhbm5lciIsICItZW5jb2RlcnMiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUpLnN0ZG91dAogICAgICAgIHJldHVybiAiaDI2NF9udmVuYyIgaW4gb3V0CiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiBGYWxzZQoKTlZFTkMgPSBoYXNfbnZlbmMoKQoKIyBDUFUgZmFsbGJhY2sgbXVzdCBzdGF5IHdhdGNoYWJsZSAqYW5kKiBmYXN0OiBDb2xhYidzIHNoYXJlZCB2Q1BVcyBhcmUgc2xvdywgc28KIyB3ZSB1c2UgYSBjaGVhcGVyIHgyNjQgcHJlc2V0LCBjYXAgdGhyZWFkcyBzZW5zaWJseSBhbmQgcmVuZGVyIGZyb20gYSBzbWFsbGVyCiMgc3VwZXJzYW1wbGUgKHNlZSBTUyBiZWxvdykuCkNQVV9DT1VOVCA9IG1heCgxLCAob3MuY3B1X2NvdW50KCkgb3IgMikpCmlmIExBTkVTIDw9IDA6CiAgICBMQU5FUyA9IDQgaWYgTlZFTkMgZWxzZSBtYXgoMiwgbWluKDQsIENQVV9DT1VOVCkpCgpHUFVfQ09ERUMgPSBbIi1jOnYiLCAiaDI2NF9udmVuYyIsICItcHJlc2V0IiwgInA0IiwgIi1yYyIsICJ2YnIiLCAiLWNxIiwgIjIzIiwgIi1iOnYiLCAiOE0iXQpDUFVfQ09ERUMgPSBbIi1jOnYiLCAibGlieDI2NCIsICItcHJlc2V0Iiwgb3MuZW52aXJvbi5nZXQoIlNXX1gyNjRfUFJFU0VUIiwgInZlcnlmYXN0IiksCiAgICAgICAgICAgICAiLWNyZiIsIG9zLmVudmlyb24uZ2V0KCJTV19DUkYiLCAiMjMiKSwgIi10aHJlYWRzIiwKICAgICAgICAgICAgIHN0cihtYXgoMSwgQ1BVX0NPVU5UIC8vIG1heCgxLCBMQU5FUykpKV0KVkNPREVDID0gR1BVX0NPREVDIGlmIE5WRU5DIGVsc2UgQ1BVX0NPREVDCgojIHN1cGVyc2FtcGxlIGZhY3RvciBiZWZvcmUgem9vbXBhbjogMnggb24gR1BVIGJveGVzLCAxLjI1eCBvbiBDUFUtb25seSBydW50aW1lcwpTUyA9IDIuMCBpZiBOVkVOQyBlbHNlIDEuMjUKCnByaW50KGYiW3NjZW5lLXdlYXZlcl0gZW5jb2RlciBtb2RlID0geydHUFUgKE5WRU5DKScgaWYgTlZFTkMgZWxzZSAnQ1BVIChsaWJ4MjY0KSd9LCAiCiAgICAgIGYibGFuZXM9e0xBTkVTfSwgY3B1cz17Q1BVX0NPVU5UfSIpCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBjaW5lbWF0b2dyYXBoeSAtLS0KCk1PVkVTID0gWwogICAgKDEuMDAsIDEuMjAsIDAuNSwgMC41LCAwLjUsIDAuNSksCiAgICAoMS4yMiwgMS4wMCwgMC41LCAwLjUsIDAuNSwgMC41KSwKICAgICgxLjE0LCAxLjE0LCAwLjAsIDEuMCwgMC41LCAwLjUpLAogICAgKDEuMTQsIDEuMTQsIDEuMCwgMC4wLCAwLjUsIDAuNSksCiAgICAoMS4xNCwgMS4xNCwgMC41LCAwLjUsIDAuMCwgMS4wKSwKICAgICgxLjE0LCAxLjE0LCAwLjUsIDAuNSwgMS4wLCAwLjApLAogICAgKDEuMDIsIDEuMjQsIDAuMywgMC4yOCwgMC4zLCAwLjIyKSwKICAgICgxLjAyLCAxLjI0LCAwLjcsIDAuNzIsIDAuNywgMC43OCksCiAgICAoMS4wOCwgMS4yMiwgMC4xNSwgMC44NSwgMC44NSwgMC4xNSksCl0KCkdSQURFUyA9IHsKICAgICJuaWdodCI6ICAoIjEuMTQiLCAiLTAuMDQ1IiwgIjEuMDUiLCAiMC4xMDowLjAyOi0wLjEwIiksCiAgICAic3Vuc2V0IjogKCIxLjEwIiwgIjAuMDIiLCAiMS4zMCIsICIwLjEwOjAuMDE6LTAuMDgiKSwKICAgICJ3YXJtIjogICAoIjEuMDgiLCAiMC4wMTUiLCAiMS4yMiIsICIwLjA2OjAuMDA6LTAuMDUiKSwKICAgICJjb29sIjogICAoIjEuMTAiLCAiMC4wIiwgIjEuMTIiLCAiLTAuMDY6MC4wMDowLjA3IiksCiAgICAidGVuc2UiOiAgKCIxLjI2IiwgIi0wLjAzIiwgIjAuOTIiLCAiMC4wNzotMC4wMjotMC4wMyIpLAogICAgInJhaW4iOiAgICgiMS4xMiIsICItMC4wMiIsICIwLjk1IiwgIi0wLjA1OjAuMDA6MC4wOCIpLAogICAgImJyaWdodCI6ICgiMS4wNiIsICIwLjAzNSIsICIxLjI4IiwgIjAuMDM6MC4wMTotMC4wMiIpLAogICAgImRyZWFtIjogICgiMS4wMiIsICIwLjAzIiwgIjEuMzQiLCAiMC4wNTotMC4wMTowLjA1IiksCn0KQ1lDTEUgPSBbImJyaWdodCIsICJ3YXJtIiwgImNvb2wiLCAiZHJlYW0iLCAidGVuc2UiXQoKS0VZUyA9IFsKICAgICgibmlnaHQiLCBbIm5pZ2h0IiwgIm1pZG5pZ2h0IiwgIm1vb24iLCAiZGFyayByb29tIiwgInN0YXJsaXQiLCAic3RyZWV0bGlnaHQiXSksCiAgICAoInN1bnNldCIsIFsic3Vuc2V0IiwgImR1c2siLCAiZ29sZGVuIGhvdXIiLCAic3VucmlzZSIsICJkYXduIiwgImZpcmUiLCAiZmxhbWUiLCAibGFudGVybiJdKSwKICAgICgicmFpbiIsIFsicmFpbiIsICJzdG9ybSIsICJ3ZXQiLCAibW9uc29vbiIsICJmb2ciLCAibWlzdCJdKSwKICAgICgidGVuc2UiLCBbImFuZ3J5IiwgImZpZ2h0IiwgImJsb29kIiwgInNjcmVhbSIsICJmZWFyIiwgInNoYWRvdyIsICJ0aHJlYXQiLCAiYmF0dGxlIl0pLAogICAgKCJicmlnaHQiLCBbInN1bmxpZ2h0IiwgInN1bm55IiwgIm1vcm5pbmciLCAibWFya2V0IiwgImZlc3RpdmFsIiwgInNtaWxlIiwgImxhdWdoIl0pLAogICAgKCJkcmVhbSIsIFsibWVtb3J5IiwgImRyZWFtIiwgImZsYXNoYmFjayIsICJza3kiLCAiaG9wZSIsICJtYWdpYyJdKSwKICAgICgid2FybSIsIFsiaW5kb29yIiwgInJvb20iLCAia2l0Y2hlbiIsICJsYW1wIiwgIndhcm0iXSksCiAgICAoImNvb2wiLCBbImNvbGQiLCAicm9vZnRvcCIsICJob3NwaXRhbCIsICJvZmZpY2UiLCAic2Nob29sIiwgInRyYWluIl0pLApdCgoKZGVmIGdyYWRlX2Zvcihwcm9tcHQsIGkpOgogICAgdCA9IChwcm9tcHQgb3IgIiIpLmxvd2VyKCkKICAgIGZvciBuYW1lLCB3b3JkcyBpbiBLRVlTOgogICAgICAgIGlmIGFueSh3IGluIHQgZm9yIHcgaW4gd29yZHMpOgogICAgICAgICAgICByZXR1cm4gR1JBREVTW25hbWVdCiAgICByZXR1cm4gR1JBREVTW0NZQ0xFW2kgJSBsZW4oQ1lDTEUpXV0KCgpkZWYgbW92ZV9mb3IoaSk6CiAgICBoID0gaW50KGhhc2hsaWIubWQ1KGYibXtpfSIuZW5jb2RlKCkpLmhleGRpZ2VzdCgpWzo4XSwgMTYpCiAgICByZXR1cm4gTU9WRVNbaCAlIGxlbihNT1ZFUyldCgoKZGVmIGNsaXBfZmlsdGVyKGksIGR1ciwgcHJvbXB0KToKICAgICIiInpvb21wYW4gS2VuIEJ1cm5zICsgY29sb3VyIGdyYWRlLCBhbHdheXMgb3V0cHV0IGV4YWN0IDE2OjkgMTA4MHAuIiIiCiAgICB6MCwgejEsIHgwLCB4MSwgeTAsIHkxID0gbW92ZV9mb3IoaSkKICAgIGZyYW1lcyA9IG1heCgxLCByb3VuZChkdXIgKiBGUFMpKQogICAgY29udHJhc3QsIGJyaWdodCwgc2F0LCBjYiA9IGdyYWRlX2Zvcihwcm9tcHQsIGkpCiAgICAjIHByb2dyZXNzIDAuLjEgYWNyb3NzIHRoZSBjbGlwLCBlYXNlZAogICAgcCA9IGYiKG9uL3ttYXgoMSwgZnJhbWVzIC0gMSl9KSIKICAgIGUgPSBmIih7cH0qe3B9KigzLTIqe3B9KSkiCiAgICB6ID0gZiIoe3owfSsoe3oxfS17ejB9KSp7ZX0pIgogICAgZnggPSBmIih7eDB9Kyh7eDF9LXt4MH0pKntlfSkiCiAgICBmeSA9IGYiKHt5MH0rKHt5MX0te3kwfSkqe2V9KSIKICAgIHJldHVybiAoCiAgICAgICAgZiJzY2FsZT17aW50KFcqU1MpfTp7aW50KEgqU1MpfTpmb3JjZV9vcmlnaW5hbF9hc3BlY3RfcmF0aW89aW5jcmVhc2UsIgogICAgICAgIGYiY3JvcD17aW50KFcqU1MpfTp7aW50KEgqU1MpfSxzZXRzYXI9MSwiCiAgICAgICAgZiJ6b29tcGFuPXo9J3t6fSc6eD0nKGl3LWl3L3pvb20pKntmeH0nOnk9JyhpaC1paC96b29tKSp7Znl9JyIKICAgICAgICBmIjpkPXtmcmFtZXN9OnM9e1d9eHtIfTpmcHM9e0ZQU30sIgogICAgICAgIGYiZXE9Y29udHJhc3Q9e2NvbnRyYXN0fTpicmlnaHRuZXNzPXticmlnaHR9OnNhdHVyYXRpb249e3NhdH0sIgogICAgICAgIGYiY29sb3JiYWxhbmNlPXJtPXtjYi5zcGxpdCgnOicpWzBdfTpnbT17Y2Iuc3BsaXQoJzonKVsxXX06Ym09e2NiLnNwbGl0KCc6JylbMl19LCIKICAgICAgICBmImZvcm1hdD15dXY0MjBwIgogICAgKQoKCmRlZiBfY2xlYW5fZXJyKGVycik6CiAgICAiIiJmZm1wZWcgcHJpbnRzIGl0cyB3aG9sZSAuL2NvbmZpZ3VyZSBsaW5lIG9uIGZhaWx1cmUg4oCUIGRyb3AgdGhlIG5vaXNlLiIiIgogICAgbGluZXMgPSBbbCBmb3IgbCBpbiAoZXJyIG9yICIiKS5zcGxpdGxpbmVzKCkKICAgICAgICAgICAgIGlmIGwuc3RyaXAoKSBhbmQgbm90IGwuc3RhcnRzd2l0aCgoIiAgY29uZmlndXJhdGlvbjoiLCAiICBsaWIiLCAiICBidWlsdCB3aXRoIikpCiAgICAgICAgICAgICBhbmQgbm90IGwuc3RhcnRzd2l0aCgiZmZtcGVnIHZlcnNpb24iKV0KICAgIHJldHVybiAiXG4iLmpvaW4obGluZXNbLTEyOl0pWy0xMjAwOl0gb3IgKGVyciBvciAiIilbLTUwMDpdCgoKZGVmIHJ1bihjbWQsIGFsbG93X2NvZGVjX2ZhbGxiYWNrPVRydWUpOgogICAgZ2xvYmFsIFZDT0RFQwogICAgciA9IHN1YnByb2Nlc3MucnVuKGNtZCwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlKQogICAgaWYgci5yZXR1cm5jb2RlID09IDA6CiAgICAgICAgcmV0dXJuCiAgICBlcnIgPSBfY2xlYW5fZXJyKHIuc3RkZXJyKQogICAgIyBOVkVOQyBjYW4gYmUgYWR2ZXJ0aXNlZCBidXQgdW51c2FibGUgKG5vIEdQVSBydW50aW1lIC8gYWxsIHNlc3Npb25zIGJ1c3kpLgogICAgaWYgYWxsb3dfY29kZWNfZmFsbGJhY2sgYW5kICJoMjY0X252ZW5jIiBpbiBjbWQ6CiAgICAgICAgcHJpbnQoZiJbc2NlbmUtd2VhdmVyXSBOVkVOQyBmYWlsZWQsIHN3aXRjaGluZyB0byBDUFUgZW5jb2Rpbmc6IHtlcnJbOjIwMF19IikKICAgICAgICBWQ09ERUMgPSBDUFVfQ09ERUMKICAgICAgICBjbWQgPSBbKCJsaWJ4MjY0IiBpZiBjID09ICJoMjY0X252ZW5jIiBlbHNlIGMpIGZvciBjIGluIGNtZF0KICAgICAgICBmb3IgZmxhZyBpbiAoIi1wcmVzZXQiLCAiLXJjIiwgIi1jcSIsICItYjp2IiwgIi10dW5lIiwgIi1jcmYiLCAiLXRocmVhZHMiKToKICAgICAgICAgICAgd2hpbGUgZmxhZyBpbiBjbWQ6CiAgICAgICAgICAgICAgICBpID0gY21kLmluZGV4KGZsYWcpCiAgICAgICAgICAgICAgICBkZWwgY21kW2k6aSArIDJdCiAgICAgICAgaSA9IGNtZC5pbmRleCgiLWM6diIpCiAgICAgICAgY21kW2kgKyAyOmkgKyAyXSA9IENQVV9DT0RFQ1syOl0KICAgICAgICByMiA9IHN1YnByb2Nlc3MucnVuKGNtZCwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlKQogICAgICAgIGlmIHIyLnJldHVybmNvZGUgPT0gMDoKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgZXJyID0gX2NsZWFuX2VycihyMi5zdGRlcnIpCiAgICByYWlzZSBSdW50aW1lRXJyb3IoZXJyKQoKCmRlZiBmZXRjaCh1cmwsIHBhdGgsIGF0dGVtcHRzPTQpOgogICAgbGFzdCA9ICIiCiAgICBmb3IgYSBpbiByYW5nZShhdHRlbXB0cyk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICByZXEgPSB1cmxsaWIucmVxdWVzdC5SZXF1ZXN0KHVybCwgaGVhZGVycz17IlVzZXItQWdlbnQiOiAic2NlbmUtd2VhdmVyLWNvbGFiIn0pCiAgICAgICAgICAgIHdpdGggdXJsbGliLnJlcXVlc3QudXJsb3BlbihyZXEsIHRpbWVvdXQ9OTApIGFzIHIsIG9wZW4ocGF0aCwgIndiIikgYXMgZjoKICAgICAgICAgICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihyLCBmKQogICAgICAgICAgICBpZiBvcy5wYXRoLmdldHNpemUocGF0aCkgPiAwOgogICAgICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgICAgIGxhc3QgPSAiZW1wdHkgZmlsZSIKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIGxhc3QgPSBzdHIoZSkKICAgICAgICB0aW1lLnNsZWVwKDAuNiAqIChhICsgMSkpCiAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJkb3dubG9hZCBmYWlsZWQ6IHtsYXN0fSIpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGpvYiAtLS0KCmRlZiBzZXRfam9iKGppZCwgKiprdyk6CiAgICB3aXRoIExPQ0s6CiAgICAgICAgSk9CU1tqaWRdLnVwZGF0ZShrdykKCgpkZWYgcmVuZGVyKGppZCwgcGFuZWxzKToKICAgIGQgPSBvcy5wYXRoLmpvaW4oV09SSywgamlkKQogICAgb3MubWFrZWRpcnMoZCwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG4gPSBsZW4ocGFuZWxzKQogICAgZG9uZSA9IFswXQoKICAgIGRlZiBvbmUoaSk6CiAgICAgICAgcCA9IHBhbmVsc1tpXQogICAgICAgIGR1ciA9IG1heCgwLjgsIGZsb2F0KHBbImVuZCJdKSAtIGZsb2F0KHBbInN0YXJ0Il0pKQogICAgICAgICMgY3Jvc3NmYWRlIG5lZWRzIFhGIGV4dHJhIHNlY29uZHMgb2YgdGFpbCBvbiBldmVyeSBjbGlwIGJ1dCB0aGUgbGFzdAogICAgICAgIHRhaWwgPSBYRiBpZiBpIDwgbiAtIDEgZWxzZSAwLjAKICAgICAgICBpbWcgPSBvcy5wYXRoLmpvaW4oZCwgZiJpe2k6MDZkfSIpCiAgICAgICAgY2xpcCA9IG9zLnBhdGguam9pbihkLCBmImN7aTowNmR9Lm1wNCIpCiAgICAgICAgZmV0Y2gocFsidXJsIl0sIGltZykKICAgICAgICBydW4oWyJmZm1wZWciLCAiLXkiLCAiLWxvb3AiLCAiMSIsICItaSIsIGltZywgIi10IiwgZiJ7ZHVyICsgdGFpbDouM2Z9IiwKICAgICAgICAgICAgICItdmYiLCBjbGlwX2ZpbHRlcihpLCBkdXIgKyB0YWlsLCBwLmdldCgicHJvbXB0IikpLAogICAgICAgICAgICAgIi1yIiwgc3RyKEZQUyksICpWQ09ERUMsICItcGl4X2ZtdCIsICJ5dXY0MjBwIiwgY2xpcF0pCiAgICAgICAgb3MucmVtb3ZlKGltZykKICAgICAgICBkb25lWzBdICs9IDEKICAgICAgICBzZXRfam9iKGppZCwgcGN0PXJvdW5kKGRvbmVbMF0gLyBuICogNzgpLAogICAgICAgICAgICAgICAgbm90ZT1mIlJlbmRlcmluZyBwYW5lbHMgb24gR1BVIMK3IHtkb25lWzBdfS97bn0iKQogICAgICAgIHJldHVybiBkdXIKCiAgICB3aXRoIFRocmVhZFBvb2xFeGVjdXRvcihtYXhfd29ya2Vycz1MQU5FUykgYXMgZXg6CiAgICAgICAgZHVycyA9IGxpc3QoZXgubWFwKG9uZSwgcmFuZ2UobikpKQoKICAgICMgLS0tLSBjcm9zcy1mYWRlIGluc2lkZSBncm91cHMsIHRoZW4gc3RyZWFtLWNvcHkgY29uY2F0IHRoZSBncm91cHMgLS0tLS0tCiAgICBncm91cHMgPSBbXQogICAgZ2kgPSAwCiAgICBmb3IgZzAgaW4gcmFuZ2UoMCwgbiwgR1JPVVApOgogICAgICAgIGlkeHMgPSBsaXN0KHJhbmdlKGcwLCBtaW4obiwgZzAgKyBHUk9VUCkpKQogICAgICAgIGdwYXRoID0gb3MucGF0aC5qb2luKGQsIGYiZ3tnaTowNWR9Lm1wNCIpCiAgICAgICAgaWYgbGVuKGlkeHMpID09IDE6CiAgICAgICAgICAgIHNodXRpbC5jb3B5KG9zLnBhdGguam9pbihkLCBmImN7aWR4c1swXTowNmR9Lm1wNCIpLCBncGF0aCkKICAgICAgICBlbHNlOgogICAgICAgICAgICBhcmdzLCBmYywgcHJldiA9IFtdLCBbXSwgIjA6diIKICAgICAgICAgICAgZm9yIGkgaW4gaWR4czoKICAgICAgICAgICAgICAgIGFyZ3MgKz0gWyItaSIsIG9zLnBhdGguam9pbihkLCBmImN7aTowNmR9Lm1wNCIpXQogICAgICAgICAgICAjIGNsaXAgayBzdGFydHMgYXQgdGhlIHN1bSBvZiB0aGUgKnZpc2libGUqIGR1cmF0aW9ucyBiZWZvcmUgaXQsCiAgICAgICAgICAgICMgYW5kIGl0cyBjcm9zcy1mYWRlIHdpdGggdGhlIHByZXZpb3VzIGNsaXAgYmVnaW5zIGV4YWN0bHkgdGhlcmUKICAgICAgICAgICAgb2ZmID0gMC4wCiAgICAgICAgICAgIGZvciBrIGluIHJhbmdlKDEsIGxlbihpZHhzKSk6CiAgICAgICAgICAgICAgICBvZmYgKz0gbWF4KDAuOCwgZHVyc1tpZHhzW2sgLSAxXV0pCiAgICAgICAgICAgICAgICBsYWIgPSBmInh7a30iCiAgICAgICAgICAgICAgICBmYy5hcHBlbmQoZiJbe3ByZXZ9XVt7a306dl14ZmFkZT10cmFuc2l0aW9uPWZhZGU6ZHVyYXRpb249e1hGfToiCiAgICAgICAgICAgICAgICAgICAgICAgICAgZiJvZmZzZXQ9e21heCgwLjA1LCBvZmYpOi4zZn1be2xhYn1dIikKICAgICAgICAgICAgICAgIHByZXYgPSBsYWIKCiAgICAgICAgICAgIHNwYW4gPSBzdW0obWF4KDAuOCwgZHVyc1tpXSkgZm9yIGkgaW4gaWR4cykKICAgICAgICAgICAgcnVuKFsiZmZtcGVnIiwgIi15IiwgKmFyZ3MsICItZmlsdGVyX2NvbXBsZXgiLCAiOyIuam9pbihmYyksCiAgICAgICAgICAgICAgICAgIi1tYXAiLCBmIlt7cHJldn1dIiwgIi10IiwgZiJ7c3BhbjouM2Z9IiwKICAgICAgICAgICAgICAgICAiLXIiLCBzdHIoRlBTKSwgKlZDT0RFQywgIi1waXhfZm10IiwgInl1djQyMHAiLCBncGF0aF0pCgogICAgICAgIGdyb3Vwcy5hcHBlbmQoZ3BhdGgpCiAgICAgICAgZ2kgKz0gMQogICAgICAgIHNldF9qb2IoamlkLCBwY3Q9NzggKyByb3VuZChnaSAvIG1heCgxLCBtYXRoLmNlaWwobiAvIEdST1VQKSkgKiAxOCksCiAgICAgICAgICAgICAgICBub3RlPWYiU3RpdGNoaW5nIMK3IHBhcnQge2dpfS97bWF0aC5jZWlsKG4gLyBHUk9VUCl9IikKCiAgICBsaXN0ZiA9IG9zLnBhdGguam9pbihkLCAibGlzdC50eHQiKQogICAgd2l0aCBvcGVuKGxpc3RmLCAidyIpIGFzIGY6CiAgICAgICAgZm9yIGcgaW4gZ3JvdXBzOgogICAgICAgICAgICBmLndyaXRlKGYiZmlsZSAne2d9J1xuIikKICAgIGZpbmFsID0gb3MucGF0aC5qb2luKE9VVCwgZiJ7amlkfS5tcDQiKQogICAgc2V0X2pvYihqaWQsIHBjdD05Nywgbm90ZT0iV3JpdGluZyBmaW5hbCBtcDTigKYiKQogICAgcnVuKFsiZmZtcGVnIiwgIi15IiwgIi1mIiwgImNvbmNhdCIsICItc2FmZSIsICIwIiwgIi1pIiwgbGlzdGYsCiAgICAgICAgICItYyIsICJjb3B5IiwgIi1tb3ZmbGFncyIsICIrZmFzdHN0YXJ0IiwgZmluYWxdKQogICAgc2h1dGlsLnJtdHJlZShkLCBpZ25vcmVfZXJyb3JzPVRydWUpCiAgICBzaXplID0gb3MucGF0aC5nZXRzaXplKGZpbmFsKQogICAgc2V0X2pvYihqaWQsIHBjdD0xMDAsIHN0YXRlPSJkb25lIiwgbm90ZT0iVmlkZW8gcmVhZHkiLCBzaXplPXNpemUsCiAgICAgICAgICAgIGRvd25sb2FkPWYiL2Rvd25sb2FkL3tqaWR9Lm1wNCIpCgoKZGVmIHdvcmtlcihqaWQsIHBhbmVscyk6CiAgICB0cnk6CiAgICAgICAgcmVuZGVyKGppZCwgcGFuZWxzKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgIHNldF9qb2IoamlkLCBzdGF0ZT0iZXJyb3IiLCBub3RlPXN0cihlKVs6NTAwXSkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gc2VydmVyIC0tLQoKQ09SUyA9IHsKICAgICJBY2Nlc3MtQ29udHJvbC1BbGxvdy1PcmlnaW4iOiAiKiIsCiAgICAiQWNjZXNzLUNvbnRyb2wtQWxsb3ctSGVhZGVycyI6ICJjb250ZW50LXR5cGUsYXV0aG9yaXphdGlvbiIsCiAgICAiQWNjZXNzLUNvbnRyb2wtQWxsb3ctTWV0aG9kcyI6ICJHRVQsUE9TVCxPUFRJT05TIiwKfQoKCmNsYXNzIEhhbmRsZXIoQmFzZUhUVFBSZXF1ZXN0SGFuZGxlcik6CiAgICBwcm90b2NvbF92ZXJzaW9uID0gIkhUVFAvMS4xIgoKICAgIGRlZiBsb2dfbWVzc2FnZShzZWxmLCAqYSk6CiAgICAgICAgcGFzcwoKICAgIGRlZiBfc2VuZChzZWxmLCBjb2RlLCBvYmosIGV4dHJhPU5vbmUpOgogICAgICAgIGJvZHkgPSBqc29uLmR1bXBzKG9iaikuZW5jb2RlKCkKICAgICAgICBzZWxmLnNlbmRfcmVzcG9uc2UoY29kZSkKICAgICAgICBmb3IgaywgdiBpbiB7KipDT1JTLCAqKihleHRyYSBvciB7fSl9Lml0ZW1zKCk6CiAgICAgICAgICAgIHNlbGYuc2VuZF9oZWFkZXIoaywgdikKICAgICAgICBzZWxmLnNlbmRfaGVhZGVyKCJDb250ZW50LVR5cGUiLCAiYXBwbGljYXRpb24vanNvbiIpCiAgICAgICAgc2VsZi5zZW5kX2hlYWRlcigiQ29udGVudC1MZW5ndGgiLCBzdHIobGVuKGJvZHkpKSkKICAgICAgICBzZWxmLmVuZF9oZWFkZXJzKCkKICAgICAgICBzZWxmLndmaWxlLndyaXRlKGJvZHkpCgogICAgZGVmIF9hdXRoKHNlbGYpOgogICAgICAgIGlmIG5vdCBUT0tFTjoKICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgICAgICByZXR1cm4gc2VsZi5oZWFkZXJzLmdldCgiQXV0aG9yaXphdGlvbiIsICIiKSA9PSBmIkJlYXJlciB7VE9LRU59IgoKICAgIGRlZiBkb19PUFRJT05TKHNlbGYpOgogICAgICAgIHNlbGYuc2VuZF9yZXNwb25zZSgyMDQpCiAgICAgICAgZm9yIGssIHYgaW4gQ09SUy5pdGVtcygpOgogICAgICAgICAgICBzZWxmLnNlbmRfaGVhZGVyKGssIHYpCiAgICAgICAgc2VsZi5lbmRfaGVhZGVycygpCgogICAgZGVmIGRvX0dFVChzZWxmKToKICAgICAgICBwYXRoID0gdXJscGFyc2Uoc2VsZi5wYXRoKS5wYXRoCiAgICAgICAgaWYgcGF0aCA9PSAiL2hlYWx0aCI6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9zZW5kKDIwMCwgeyJvayI6IFRydWUsICJncHUiOiBOVkVOQywgImxhbmVzIjogTEFORVMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJtb2RlIjogImdwdSIgaWYgTlZFTkMgZWxzZSAiY3B1In0pCiAgICAgICAgbSA9IHJlLm1hdGNoKHIiXi9zdGF0dXMvKFtcdy1dKykkIiwgcGF0aCkKICAgICAgICBpZiBtOgogICAgICAgICAgICBqID0gSk9CUy5nZXQobS5ncm91cCgxKSkKICAgICAgICAgICAgcmV0dXJuIHNlbGYuX3NlbmQoMjAwLCBqKSBpZiBqIGVsc2Ugc2VsZi5fc2VuZCg0MDQsIHsiZXJyb3IiOiAibm8gam9iIn0pCiAgICAgICAgbSA9IHJlLm1hdGNoKHIiXi9kb3dubG9hZC8oW1x3LV0rKVwubXA0JCIsIHBhdGgpCiAgICAgICAgaWYgbToKICAgICAgICAgICAgZiA9IG9zLnBhdGguam9pbihPVVQsIGYie20uZ3JvdXAoMSl9Lm1wNCIpCiAgICAgICAgICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhmKToKICAgICAgICAgICAgICAgIHJldHVybiBzZWxmLl9zZW5kKDQwNCwgeyJlcnJvciI6ICJub3QgcmVhZHkifSkKICAgICAgICAgICAgc2l6ZSA9IG9zLnBhdGguZ2V0c2l6ZShmKQogICAgICAgICAgICBzZWxmLnNlbmRfcmVzcG9uc2UoMjAwKQogICAgICAgICAgICBmb3IgaywgdiBpbiBDT1JTLml0ZW1zKCk6CiAgICAgICAgICAgICAgICBzZWxmLnNlbmRfaGVhZGVyKGssIHYpCiAgICAgICAgICAgIHNlbGYuc2VuZF9oZWFkZXIoIkNvbnRlbnQtVHlwZSIsICJ2aWRlby9tcDQiKQogICAgICAgICAgICBzZWxmLnNlbmRfaGVhZGVyKCJDb250ZW50LUxlbmd0aCIsIHN0cihzaXplKSkKICAgICAgICAgICAgc2VsZi5zZW5kX2hlYWRlcigiQ29udGVudC1EaXNwb3NpdGlvbiIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZidhdHRhY2htZW50OyBmaWxlbmFtZT0ibWFuZ2EtdmlkZW8ubXA0IicpCiAgICAgICAgICAgIHNlbGYuZW5kX2hlYWRlcnMoKQogICAgICAgICAgICB3aXRoIG9wZW4oZiwgInJiIikgYXMgZmg6CiAgICAgICAgICAgICAgICBzaHV0aWwuY29weWZpbGVvYmooZmgsIHNlbGYud2ZpbGUsIDEwMjQgKiAxMDI0KQogICAgICAgICAgICByZXR1cm4KICAgICAgICBzZWxmLl9zZW5kKDQwNCwgeyJlcnJvciI6ICJub3QgZm91bmQifSkKCiAgICBkZWYgZG9fUE9TVChzZWxmKToKICAgICAgICBpZiB1cmxwYXJzZShzZWxmLnBhdGgpLnBhdGggIT0gIi9yZW5kZXIiOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fc2VuZCg0MDQsIHsiZXJyb3IiOiAibm90IGZvdW5kIn0pCiAgICAgICAgaWYgbm90IHNlbGYuX2F1dGgoKToKICAgICAgICAgICAgcmV0dXJuIHNlbGYuX3NlbmQoNDAxLCB7ImVycm9yIjogImJhZCB0b2tlbiJ9KQogICAgICAgIG4gPSBpbnQoc2VsZi5oZWFkZXJzLmdldCgiQ29udGVudC1MZW5ndGgiLCAiMCIpKQogICAgICAgIHRyeToKICAgICAgICAgICAgZGF0YSA9IGpzb24ubG9hZHMoc2VsZi5yZmlsZS5yZWFkKG4pIG9yIGIie30iKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9zZW5kKDQwMCwgeyJlcnJvciI6ICJiYWQganNvbiJ9KQogICAgICAgIHBhbmVscyA9IGRhdGEuZ2V0KCJwYW5lbHMiKSBvciBbXQogICAgICAgIGlmIG5vdCBwYW5lbHM6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9zZW5kKDQwMCwgeyJlcnJvciI6ICJubyBwYW5lbHMifSkKICAgICAgICBqaWQgPSB1dWlkLnV1aWQ0KCkuaGV4WzoxMl0KICAgICAgICB3aXRoIExPQ0s6CiAgICAgICAgICAgIEpPQlNbamlkXSA9IHsiaWQiOiBqaWQsICJzdGF0ZSI6ICJydW5uaW5nIiwgInBjdCI6IDAsCiAgICAgICAgICAgICAgICAgICAgICAgICAibm90ZSI6ICJRdWV1ZWQiLCAicGFuZWxzIjogbGVuKHBhbmVscyl9CiAgICAgICAgdGhyZWFkaW5nLlRocmVhZCh0YXJnZXQ9d29ya2VyLCBhcmdzPShqaWQsIHBhbmVscyksIGRhZW1vbj1UcnVlKS5zdGFydCgpCiAgICAgICAgc2VsZi5fc2VuZCgyMDAsIHsiaWQiOiBqaWR9KQoKCmRlZiBzZXJ2ZShwb3J0PTgwMDApOgogICAgVGhyZWFkaW5nSFRUUFNlcnZlcigoIjAuMC4wLjAiLCBwb3J0KSwgSGFuZGxlcikuc2VydmVfZm9yZXZlcigpCg=="
    open('/content/encoder_server.py', 'wb').write(base64.b64decode(_B64))

import importlib.util
spec = importlib.util.spec_from_file_location('encoder_server', '/content/encoder_server.py')
enc = importlib.util.module_from_spec(spec); spec.loader.exec_module(enc)
threading.Thread(target=enc.serve, kwargs={'port': 8000}, daemon=True).start()
time.sleep(2)
print('encoder running · GPU NVENC =', enc.NVENC)

# Colab does not include cloudflared. Download the standalone binary automatically.
cloudflared = shutil.which('cloudflared')
if not cloudflared:
    machine = platform.machine().lower()
    arch = 'arm64' if machine in ('aarch64', 'arm64') else 'amd64'
    cloudflared = '/content/cloudflared'
    binary_url = f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}'
    print('installing cloudflared…')
    req = urllib.request.Request(binary_url, headers={'User-Agent': 'scene-weaver-colab'})
    with urllib.request.urlopen(req, timeout=120) as r, open(cloudflared, 'wb') as f:
        f.write(r.read())
    os.chmod(cloudflared, 0o755)
    if os.path.getsize(cloudflared) < 1_000_000:
        raise RuntimeError('cloudflared download was incomplete; re-run this cell')
print('cloudflared ready:', cloudflared)

p = subprocess.Popen([cloudflared,'tunnel','--url','http://localhost:8000','--no-autoupdate'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
url = None
for line in p.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\n' + '='*64)
print('PASTE THIS INTO THE APP:')
print(url or 'tunnel failed — re-run this cell')
print('='*64)


In [ ]:
#@title 2 · Keep alive (leave running while the video encodes)
import time
while True:
    time.sleep(60)